# A5. Dimensionally constrained lead equations against their free-form counterparts (Table S2) and what each ozone term does

2026-10-20 (corrected fire file; every search re-run). Both lead equations come from the dimensionally constrained front:
  PM2.5, complexity 13, seed 0:  PM2.5 = sqrt(a (F_E,-2 + F_N)) + b CO / sqrt(u)
  O3,    complexity 15, seed 0 of the front with PM2.5 and T offered:  O3 = sqrt(a SW sqrt(PM2.5 NO2 / NO) + b F_E,-2)
This notebook gives (1) Table S2: each lead against its free-form counterpart and against the variants the text discusses
(O3 searched without PM2.5 offered; PM2.5 searched with ozone offered), all scored the same way on identical days;
(2) ablations of the ozone lead (what PM2.5, fire and the NO2/NO ratio each contribute); (3) elasticities and the
PM2.5 effect across temperature and fire terciles; (4) the split of the ozone prediction between its two terms;
(5) the three-seed stability of the ozone search with and without PM2.5 offered.
Scoring: leave-one-year-out, constants refitted on the training years, pooled R2 and mean of per-year R2.

In [1]:
# Setup
import sys
from pathlib import Path
HERE = Path.cwd().resolve()
sys.path.insert(0, str(HERE.parent)); sys.path.insert(0, str(HERE.parents[1]))
import warnings
import numpy as np, pandas as pd
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
from style import ROOT, DATA_OUT
from termmap_utils import load_front, term_presence, term_summary
warnings.filterwarnings("ignore")
B = ROOT / "Data/processed/colombia/bogota"

In [2]:
# Settings (edit here)
O3_LEAD_FRONT = B / "bogota_o3_pysr_units_fire2_pm_pareto_front_seed0.csv"   # front of the O3 lead equation (PM2.5 and T offered)
SEED_TABLES = {"PM2.5 and T offered": B / "bogota_o3_pysr_units_fire2_pm_scored.csv",
               "PM2.5 not offered (T offered)": B / "bogota_o3_pysr_units_fire2_scored.csv"}
SAVE = True

In [3]:
# Models (constants refitted; p0 = values printed by PySR)
def pm25_lead(X, a, b):                 # c13 seed 0
    co, u, fn, fe2 = X; return np.sqrt(np.clip(a * (fe2 + fn), 0, None)) + b * co / np.sqrt(u)
def pm25_free(X, k1, k2, k3):           # free-form front, c13: F_NE,-2/k1 + k3 (CO + (k2 - w700)/u)
    co, w700, u, ne2 = X; return ne2 / k1 + k3 * (co + (k2 - w700) / u)
def pm25_o3(X, a, b, c):                # PM2.5 with ozone offered, c13 seed 1: b CO + a F_E,-2 + O3/(c u)
    co, u, fe2, o3 = X; return a * fe2 + b * co + o3 / (c * u)
def o3_lead(X, a, b):                   # c15 seed 0
    no, no2, sw, pm, fe2 = X; return np.sqrt(np.clip(a * sw * np.sqrt(np.clip(pm * no2 / no, 0, None)) + b * fe2, 0, None))
def o3_nopm(X, c, d):                   # PM2.5 not offered, c15 seed 1: F/c + d SW sqrt(NO2/(NO u))
    no, no2, sw, fe2, u = X; return np.sqrt(np.clip(fe2 / c + d * sw * np.sqrt(no2 / (no * u)), 0, None))
def o3_free(X, k):                      # free form with T and PM2.5 offered, c13: sqrt(k sqrt(PM2.5 NO2/NO) (F + T SW))
    no, no2, sw, pm, fe2, t = X; return np.sqrt(np.clip(k * np.sqrt(np.clip(pm * no2 / no, 0, None)) * (fe2 + t * sw), 0, None))
def o3_no_pm(X, a, b, c):               # ablation: PM2.5 replaced by a free constant
    no, no2, sw, pm, fe2 = X; return np.sqrt(np.clip(a * sw * np.sqrt(c * no2 / no) + b * fe2, 0, None))
def o3_no_fire(X, a):                   # ablation: fire term removed
    no, no2, sw, pm, fe2 = X; return np.sqrt(np.clip(a * sw * np.sqrt(np.clip(pm * no2 / no, 0, None)), 0, None))
def o3_no_ratio(X, a, b, c):            # ablation: NO2/NO replaced by a free constant
    no, no2, sw, pm, fe2 = X; return np.sqrt(np.clip(a * sw * np.sqrt(c * pm) + b * fe2, 0, None))

def load(name):
    d = pd.read_parquet(B / name); d["year"] = pd.to_datetime(d["date"]).dt.year; return d
PM_F = ["co", "wind_speed_era5", "hotspots_sector_N", "hotspots_sector_E_lag2"]
O3_POOL = ["no2", "no", "radiation_mean", "hotspots_sector_E_lag2", "rh_2m", "wind_speed_era5", "w700", "temp_2m", "pm25"]   # identical evaluation days
dp = load("bogota_master_dataset.parquet").dropna(subset=PM_F + ["w700", "hotspots_sector_NE_lag2", "pm25"]).reset_index(drop=True)
do = load("bogota_o3_master_dataset.parquet").dropna(subset=O3_POOL + ["o3"]).reset_index(drop=True)
dw = load("bogota_master_dataset_with_o3.parquet").dropna(subset=PM_F + ["o3_mda8", "pm25"]).reset_index(drop=True)
print("PM2.5 days", len(dp), "| PM2.5 days with ozone", len(dw), "| O3 days", len(do))

def loyo(model, d, feats, target, p0):
    X = tuple(d[c].to_numpy(float) for c in feats); y = d[target].to_numpy(float); yrs = d.year.to_numpy()
    pf, _ = curve_fit(model, X, y, p0=p0, maxfev=50000)
    pred = np.full(len(y), np.nan)
    for yr in np.unique(yrs):
        te = yrs == yr
        try: p, _ = curve_fit(model, tuple(a[~te] for a in X), y[~te], p0=pf, maxfev=50000)
        except RuntimeError: p = pf
        pred[te] = model(tuple(a[te] for a in X), *p)
    per = float(np.mean([r2_score(y[yrs == yr], pred[yrs == yr]) for yr in np.unique(yrs)]))
    return dict(pooled=r2_score(y, pred), mean_of_years=per, in_sample=r2_score(y, model(X, *pf)), n=len(y), constants=pf)

PM2.5 days 3283 | PM2.5 days with ozone 3283 | O3 days 2917


In [4]:
# 1. Table S2
PMF_FREE = ["co", "w700", "wind_speed_era5", "hotspots_sector_NE_lag2"]
O3F = ["no", "no2", "radiation_mean", "pm25", "hotspots_sector_E_lag2"]
FORMS = [("PM2.5", "Dimensionally constrained, complexity 13 (Eq. 1, reported)", pm25_lead, dp, PM_F, "pm25", [0.10428376, 15.679531]),
         ("PM2.5", "Free-form, complexity 13", pm25_free, dp, PMF_FREE, "pm25", [209.5111, 0.16998222, 15.929433]),
         ("PM2.5", "Dimensionally constrained, ozone offered, complexity 13 (seed 1), days with ozone", pm25_o3, dw, ["co", "wind_speed_era5", "hotspots_sector_E_lag2", "o3_mda8"], "pm25", [0.005585467, 15.932973, 4.7663608]),
         ("PM2.5", "Lead equation (Eq. 1) on the same days with ozone", pm25_lead, dw, PM_F, "pm25", [0.10428376, 15.679531]),
         ("O3", "Dimensionally constrained, PM2.5 and T offered, complexity 15 (Eq. 2, reported)", o3_lead, do, O3F, "o3", [1 / 2.034343, 0.10787038]),
         ("O3", "Dimensionally constrained, PM2.5 not offered, complexity 15 (seed 1)", o3_nopm, do, ["no", "no2", "radiation_mean", "hotspots_sector_E_lag2", "wind_speed_era5"], "o3", [3.8654706, 1.7302341]),
         ("O3", "Free-form, PM2.5 and T offered, complexity 13", o3_free, do, O3F + ["temp_2m"], "o3", [0.029881364])]
rows, FITS = [], {}
for pol, name, m, d, f, t, p0 in FORMS:
    r = loyo(m, d, f, t, p0); FITS[name] = r["constants"]
    rows.append(dict(pollutant=pol, form=name, n=r["n"], r2_pooled=r["pooled"], r2_mean_of_years=r["mean_of_years"],
                     r2_in_sample=r["in_sample"], constants=", ".join(f"{v:.4g}" for v in r["constants"])))
tS2 = pd.DataFrame(rows)
if SAVE: tS2.to_csv(DATA_OUT / "tableS2_constrained_vs_freeform.csv", index=False)
print(tS2.round(3).to_string(index=False))

pollutant                                                                              form    n  r2_pooled  r2_mean_of_years  r2_in_sample              constants
    PM2.5                        Dimensionally constrained, complexity 13 (Eq. 1, reported) 3283      0.677             0.655         0.685         0.09794, 16.13
    PM2.5                                                          Free-form, complexity 13 3283      0.672             0.653         0.687    220.7, 0.1584, 16.3
    PM2.5 Dimensionally constrained, ozone offered, complexity 13 (seed 1), days with ozone 3283      0.682             0.665         0.700 0.005313, 15.38, 4.443
    PM2.5                                 Lead equation (Eq. 1) on the same days with ozone 3283      0.677             0.655         0.685         0.09794, 16.13
       O3   Dimensionally constrained, PM2.5 and T offered, complexity 15 (Eq. 2, reported) 2917      0.603             0.536         0.619          0.4478, 0.161
       O3             

In [5]:
# 2. Ablations of the ozone lead (each refitted)
a0, b0 = FITS[FORMS[4][1]]
abl = []
def addo(name, m, p0):
    r = loyo(m, do, O3F, "o3", p0); abl.append(dict(variant=name, r2_pooled=r["pooled"], r2_mean_of_years=r["mean_of_years"], in_sample=r["in_sample"], constants=", ".join(f"{v:.4g}" for v in r["constants"])))
addo("lead (Eq. 2)", o3_lead, [a0, b0])
addo("PM2.5 replaced by a free constant", o3_no_pm, [a0 * 0.2, b0, 20.0])
addo("fire term removed", o3_no_fire, [a0])
addo("NO2/NO replaced by a free constant", o3_no_ratio, [a0, b0, 1.0])
abl = pd.DataFrame(abl); print(abl.round(3).to_string(index=False))
# the sunlight-only skeleton and the pieces added one at a time
def sw_only(X, a): return a * X[2]
def sw_ratio(X, a): no, no2, sw, pm, fe2 = X; return np.sqrt(np.clip(a * sw * np.sqrt(no2 / no), 0, None))
def sw_ratio_pm(X, a): return o3_no_fire(X, a)
def sw_ratio_fire(X, a, b): no, no2, sw, pm, fe2 = X; return np.sqrt(np.clip(a * sw * np.sqrt(no2 / no) + b * fe2, 0, None))
build = []
for nm, m, p0 in [("sunlight only", sw_only, [0.1]), ("sunlight and NO2/NO", sw_ratio, [1.0]), ("sunlight, NO2/NO and PM2.5", sw_ratio_pm, [a0]),
                  ("sunlight, NO2/NO and fire", sw_ratio_fire, [1.0, 0.1]), ("sunlight, NO2/NO, PM2.5 and fire (Eq. 2)", o3_lead, [a0, b0])]:
    r = loyo(m, do, O3F, "o3", p0); build.append(dict(model=nm, r2_pooled=r["pooled"], r2_mean_of_years=r["mean_of_years"]))
build = pd.DataFrame(build); print(build.round(3).to_string(index=False))

                           variant  r2_pooled  r2_mean_of_years  in_sample             constants
                      lead (Eq. 2)      0.603             0.536      0.619         0.4478, 0.161
 PM2.5 replaced by a free constant      0.546             0.480      0.559 0.4439, 0.2738, 14.87
                 fire term removed      0.563             0.491      0.580                0.5042
NO2/NO replaced by a free constant      0.382             0.277      0.424  0.4148, 0.245, 1.115
                                   model  r2_pooled  r2_mean_of_years
                           sunlight only      0.124             0.002
                     sunlight and NO2/NO      0.413             0.336
              sunlight, NO2/NO and PM2.5      0.563             0.491
               sunlight, NO2/NO and fire      0.546             0.480
sunlight, NO2/NO, PM2.5 and fire (Eq. 2)      0.603             0.536


In [6]:
# 3. Elasticities and the PM2.5 effect across temperature and fire terciles (full-data constants)
X = tuple(do[c].to_numpy(float) for c in O3F); idx = {c: i for i, c in enumerate(O3F)}
def deriv(model, p, var, rel=1e-3):
    i = idx[var]; h = np.abs(X[i]) * rel + 1e-9
    up = list(X); dn = list(X); up[i] = X[i] + h; dn[i] = X[i] - h
    return (model(tuple(up), *p) - model(tuple(dn), *p)) / (2 * h)
pred = o3_lead(X, a0, b0); el = []
for v in ("pm25", "no", "no2", "radiation_mean", "hotspots_sector_E_lag2"):
    e = deriv(o3_lead, [a0, b0], v) * X[idx[v]] / pred; el.append(dict(variable=v, median_elasticity=float(np.nanmedian(e)), mean_elasticity=float(np.nanmean(e))))
el = pd.DataFrame(el); inter = []
fire_t = do.hotspots_sector_E_lag2
for by, lab in (("temp_2m", "temperature"), ("hotspots_sector_E_lag2", "fire")):
    q = pd.qcut(do[by], 3, labels=["low", "mid", "high"])
    dv = pd.Series(deriv(o3_lead, [a0, b0], "pm25")).groupby(q.values, observed=True).median()
    inter.append(dict(derivative="dO3/dPM2.5", across_terciles_of=lab, low=dv["low"], mid=dv["mid"], high=dv["high"]))
inter = pd.DataFrame(inter); print(el.round(3).to_string(index=False)); print(inter.round(3).to_string(index=False))

              variable  median_elasticity  mean_elasticity
                  pm25              0.237            0.227
                    no             -0.237           -0.227
                   no2              0.237            0.227
        radiation_mean              0.474            0.454
hotspots_sector_E_lag2              0.026            0.046
derivative across_terciles_of   low   mid  high
dO3/dPM2.5        temperature 0.242 0.275 0.277
dO3/dPM2.5               fire 0.311 0.277 0.210


In [7]:
# 4. How the prediction splits between the photochemical term and the fire term (inside the square root, and in ppb)
Pt = a0 * do.radiation_mean * np.sqrt(do.pm25 * do.no2 / do.no); Ft = b0 * do.hotspots_sector_E_lag2; S = Pt + Ft
share = dict(photochem_share_of_radicand_median=float((Pt / S).median()), fire_share_of_radicand_median=float((Ft / S).median()), fire_share_of_radicand_p90=float((Ft / S).quantile(0.9)),
             share_variance_radicand_photochem=float(np.cov(Pt, S)[0, 1] / np.var(S)), share_variance_radicand_fire=float(np.cov(Ft, S)[0, 1] / np.var(S)),
             corr_pm25_fire=float(do[["pm25", "hotspots_sector_E_lag2"]].corr().iloc[0, 1]), corr_pm25_ratio=float(np.corrcoef(do.pm25, do.no2 / do.no)[0, 1]))
lp = np.log(np.clip(Pt, 1e-9, None)); v = np.var(lp)
comp = {"sunlight, ln SW": np.log(do.radiation_mean), "PM2.5, 1/2 ln PM2.5": 0.5 * np.log(do.pm25), "NO2/NO, 1/2 ln(NO2/NO)": 0.5 * np.log(do.no2 / do.no)}
dec = pd.DataFrame([dict(component=k, share_of_log_variance_photochem_term=float(np.cov(c, lp)[0, 1] / v)) for k, c in comp.items()])
print({k: round(x, 3) for k, x in share.items()}); print(dec.round(3).to_string(index=False), "| sum", round(dec.iloc[:, 1].sum(), 3))

{'photochem_share_of_radicand_median': 0.948, 'fire_share_of_radicand_median': 0.052, 'fire_share_of_radicand_p90': 0.24, 'share_variance_radicand_photochem': 0.777, 'share_variance_radicand_fire': 0.223, 'corr_pm25_fire': 0.521, 'corr_pm25_ratio': 0.057}
             component  share_of_log_variance_photochem_term
       sunlight, ln SW                                 0.332
   PM2.5, 1/2 ln PM2.5                                 0.283
NO2/NO, 1/2 ln(NO2/NO)                                 0.385 | sum 1.0


In [8]:
# 5. Where each variable enters the lead front, and the three-seed stability with and without PM2.5 offered
summ = term_summary(term_presence(load_front(O3_LEAD_FRONT)))
seed_rows = []
for tag, p in SEED_TABLES.items():
    s = pd.read_csv(p); s = s[s.seed.astype(str).isin(["0", "1", "2"])]
    for seed, g in s.groupby("seed"):
        for lab, gg in (("complexity <= 16", g[g.complexity <= 16]), ("any complexity", g)):
            bb = gg.loc[gg.r2_pooled.idxmax()]
            seed_rows.append(dict(pool=tag, seed=int(seed), which=lab, complexity=int(bb.complexity), r2_pooled=bb.r2_pooled, r2_mean_of_years=bb.r2_mean_years,
                                  has_pm25="pm25" in bb.equation, has_T="temp_2m" in bb.equation, has_fire="hotspots" in bb.equation, has_ratio="no2" in bb.equation and "no" in bb.equation))
seeds = pd.DataFrame(seed_rows); print(summ.to_string(index=False)); print(seeds.round(3).to_string(index=False))

              variable  first_complexity  last_complexity  n_levels_present  n_levels_from_first  frac_present_from_first  continuous_to_top
        radiation_mean                 3               23                12                   16                 0.750000              False
                  pm25                 4               23                12                   15                 0.800000              False
hotspots_sector_E_lag2                 5               23                 9                   14                 0.642857              False
                   no2                 8               23                10                   11                 0.909091              False
                    no                 9               23                10                   10                 1.000000               True
                 rh_2m                14               18                 2                    7                 0.285714              False
             

In [9]:
# Save
if SAVE:
    abl.to_csv(DATA_OUT / "o3_lead_ablations.csv", index=False); build.to_csv(DATA_OUT / "o3_lead_buildup.csv", index=False)
    el.to_csv(DATA_OUT / "o3_elasticities.csv", index=False); inter.to_csv(DATA_OUT / "o3_interaction_by_tercile.csv", index=False)
    pd.DataFrame([share]).to_csv(DATA_OUT / "o3_term_shares.csv", index=False); dec.to_csv(DATA_OUT / "term_contributions_o3_logspace.csv", index=False)
    summ.to_csv(DATA_OUT / "o3_lead_front_term_summary.csv", index=False); seeds.to_csv(DATA_OUT / "o3_constrained_seeds.csv", index=False)